# Lamina Tubulin Biexponential Fit Workflow

Clean workflow for loading a calibrated HDF5 file, preparing aligned summed decays, generating per-pixel biexponential fit maps, and visualizing the effective lifetime result.


In [ ]:
from pathlib import Path
import importlib
import sys

ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SRC = str(ROOT / "src")
if SRC in sys.path:
    sys.path.remove(SRC)
sys.path.insert(0, SRC)
MCS_FILE_SRC = ROOT.parent / "BrightEyes-MCS-File" / "src"
if MCS_FILE_SRC.exists():
    MCS_FILE_SRC = str(MCS_FILE_SRC)
    if MCS_FILE_SRC in sys.path:
        sys.path.remove(MCS_FILE_SRC)
    sys.path.insert(0, MCS_FILE_SRC)

import h5py
import matplotlib.pyplot as plt
import numpy as np

import brighteyes_mcs_file.alignment as alignment_module
import brighteyes_flim.graph_tools as graph
alignment_module = importlib.reload(alignment_module)
graph = importlib.reload(graph)

from brighteyes_mcs_file.alignment import Alignment
from brighteyes_mcs_file import show_h5_structure_html, sum_channel_applying_shifts

EXPECTED_H5_DATA_FORMAT_VERSION = "0.0.6"


def assert_h5_schema_version(filename, expected=EXPECTED_H5_DATA_FORMAT_VERSION):
    with h5py.File(filename, "r") as hf:
        actual = hf.attrs["data_format_version"]
    actual = actual.decode() if isinstance(actual, bytes) else str(actual)
    if actual != expected:
        raise ValueError(f"Expected H5 schema version {expected}, found {actual} in {filename}")
    return actual


In [ ]:
DATA_CALIBRATE = '/mnt/DATA/Mixed Data/lamina+tubulin/data-10-04-2024-19-27-47_calib.h5'
h5_schema_version = assert_h5_schema_version(DATA_CALIBRATE)


## Inspect Calibration File


In [ ]:
#show_h5_structure_html(DATA_CALIBRATE)


## Load Data And Metadata


In [ ]:
with h5py.File(DATA_CALIBRATE, "r") as hf:
    calibration = hf["calibration/results/spad"]
    metadata = hf["raw/metadata"].attrs

    laser_frequency_mhz = float(calibration.attrs["laser_frequency_mhz"])
    laser_period_ns = float(calibration.attrs["laser_period_ns"])
    nbin = int(metadata["time_bins"])
    pixel_size_x_um = float(metadata["pixel_size_x_um"])
    pxdwelltime = float(metadata["pixel_dwell_time_us"])

    data_input = hf["raw/spad"][:]
    channel_skew = calibration["timing/channel_skew_bins"][:]
    aligned_irf_trace = calibration["aligned/irf_trace"][:]

t = np.arange(nbin, dtype=float) * laser_period_ns / nbin

print(f"Using calibrated laser timing: {laser_frequency_mhz:.4f} MHz ({laser_period_ns:.4f} ns)")
print("data_input:", data_input.shape)
print("aligned_irf_trace:", aligned_irf_trace.shape)
print("pixel_size_x_um:", pixel_size_x_um)
print("pxdwelltime:", pxdwelltime)


## Sum Channels With The Calibration Shifts


In [ ]:
def spatial_bin_yx_sum(data, bin_size=(1, 1), crop=True):
    """Sum spatial blocks along y,x for data shaped (rep, z, y, x, t, ch)."""
    data = np.asarray(data)
    if data.ndim != 6:
        raise ValueError(f"expected data shape (rep, z, y, x, t, ch), got {data.shape}")

    if np.isscalar(bin_size):
        bin_y = bin_x = int(bin_size)
    else:
        bin_y, bin_x = map(int, bin_size)
    if bin_y < 1 or bin_x < 1:
        raise ValueError("bin_size values must be positive integers")

    rep, z, y, x, t_bins, ch = data.shape
    if crop:
        y_use = (y // bin_y) * bin_y
        x_use = (x // bin_x) * bin_x
        data = data[:, :, :y_use, :x_use, :, :]
    elif y % bin_y or x % bin_x:
        raise ValueError(
            f"y,x shape {(y, x)} is not divisible by bin_size {(bin_y, bin_x)}; "
            "use crop=True or choose a divisor"
        )

    rep, z, y, x, t_bins, ch = data.shape
    return data.reshape(rep, z, y // bin_y, bin_y, x // bin_x, bin_x, t_bins, ch).sum(axis=(3, 5))


# Set to (2, 2), (4, 4), etc. to merge neighboring pixels before fitting.
spatial_bin_yx = (4, 4)
if np.isscalar(spatial_bin_yx):
    spatial_bin_y = spatial_bin_x = int(spatial_bin_yx)
else:
    spatial_bin_y, spatial_bin_x = map(int, spatial_bin_yx)
spatial_bin_yx = (spatial_bin_y, spatial_bin_x)

data_input_binned = spatial_bin_yx_sum(data_input, spatial_bin_yx)
pixel_size_x_um_binned = pixel_size_x_um * spatial_bin_x
pxdwelltime_binned = pxdwelltime * spatial_bin_y * spatial_bin_x

print("data_input_binned:", data_input_binned.shape)
print("spatial_bin_yx:", spatial_bin_yx)
print("pixel_size_x_um_binned:", pixel_size_x_um_binned)
print("pxdwelltime_binned:", pxdwelltime_binned)


In [ ]:
irf_trace = np.asarray(aligned_irf_trace, dtype=float)


In [ ]:
data_summed = sum_channel_applying_shifts(data_input_binned, channel_skew, axis=())[0, 0, ...]
irf_summed = sum_channel_applying_shifts(irf_trace, channel_skew, axis=())

data_summed_rev = sum_channel_applying_shifts(
    data_input_binned,
    channel_skew,
    axis=(),
    reverse_shifts=False,
)[0, 0, ...]
irf_summed_rev = sum_channel_applying_shifts(
    irf_trace,
    channel_skew,
    axis=(),
    reverse_shifts=False,
)

data_summed_no_alignment = np.sum(data_input_binned, axis=(0, 1, 2, 3, 5))
irf_summed_no_alignment = np.sum(irf_trace, axis=-1)

print("data_summed:", data_summed.shape)
print("irf_summed:", irf_summed.shape)


In [ ]:
graph.plot_channel_skew_correction(
    irf_no_alignment=irf_summed_no_alignment,
    irf_aligned=irf_summed,
    irf_reversed=irf_summed_rev,
    data_no_alignment=data_summed_no_alignment,
    data_aligned=data_summed.sum(axis=(0, 1)),
    data_reversed=data_summed_rev.sum(axis=(0, 1)),
)


## Check One Pixel Fit


In [ ]:
preview_y = data_summed.shape[0] // 2
preview_x = data_summed.shape[1] // 2
preview_hist = data_summed[preview_y, preview_x, :]

preview_result, preview_cov = Alignment.perform_fit_data(
    t=t,
    data=preview_hist,
    irf=irf_summed,
    period=laser_period_ns,
    initial_tau=None,
    initial_dT=None,
    initial_C=None,
    mode='model_shift',
    fit_type='likelihood',
    force_C_normalized=True,
)

fit_preview = Alignment.fit_model_data(
    t,
    preview_result["C"],
    preview_result["dT"],
    preview_result["tau"],
    irf=irf_summed,
    period=laser_period_ns,
    mode='model_shift',
)

fit_preview = Alignment.to_numpy_1d(fit_preview, dtype=float)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(t, preview_hist / preview_hist.sum(), label="Data", color="black")
ax.plot(t, irf_summed / irf_summed.sum(), label="IRF", color="tab:blue")
ax.plot(t, fit_preview / fit_preview.sum(), label="Fit", color="tab:red")
ax.set_yscale("log")
ax.set_xlabel("Time (ns)")
ax.set_ylabel("Normalized counts")
ax.set_title(f"Preview pixel y={preview_y}, x={preview_x}; tau={preview_result['tau']:.2f} ns")
ax.legend()
preview_result


## Generate Biexponential Fit Maps


In [ ]:
if not hasattr(Alignment, "generate_fit_maps"):
    raise RuntimeError(
        "The notebook kernel still has an old brighteyes_mcs_file.alignment module loaded. "
        "Rerun the first import cell, or restart the kernel and run from the top."
    )


def biexponential_model(t, irf, period, C, dT, a1, tau1, delta_tau):
    tau2 = tau1 + delta_tau
    first = Alignment.fit_model_data(t, 1.0, dT, tau1, irf, period, mode="model_shift")
    second = Alignment.fit_model_data(t, 1.0, dT, tau2, irf, period, mode="model_shift")
    first = first / first.sum()
    second = second / second.sum()
    model = a1 * first + (1.0 - a1) * second
    model_sum = model.sum()
    if not np.isfinite(model_sum) or model_sum <= 0:
        return np.zeros_like(t, dtype=float)
    return C * model / model_sum


biexponential_parameter_names = ["C", "dT", "a1", "tau1", "delta_tau"]
biexponential_p0 = [1.0, 0.0, 0.5, 1.5, 2.5]
biexponential_bounds = (
    [0.0, -nbin / 2, 0.0, 0.05, 0.001],
    [np.inf, nbin / 2, 1.0, laser_period_ns/3, laser_period_ns/3],
)

fit_maps = Alignment.generate_fit_maps(
    data=data_summed,
    irf=irf_summed,
    t=t,
    period=laser_period_ns,
    initial_tau=None,
    initial_dT=None,
    initial_C=None,
    mode="model_shift",
    fit_type="likelihood",
    model_fn=biexponential_model,
    p0=biexponential_p0,
    bounds=biexponential_bounds,
    parameter_names=biexponential_parameter_names,
    lifetime_param="tau1",
    n_jobs=-1,
    job_chunk_size=10,
)

fit_stack, fit_stack_names = Alignment.fit_maps_to_stack(fit_maps)
print("fit_stack shape:", fit_stack.shape)
print("fit_stack axis 0:", fit_stack_names)

np.savez("fit_maps_biexponential.npz", **fit_maps)
print("saved fit_maps_biexponential.npz")



## Visualize Effective Lifetime Fit


In [ ]:
tau1 = fit_maps["tau1"]
tau2 = fit_maps["tau1"] + fit_maps["delta_tau"]
a1 = np.clip(fit_maps["a1"], 0.0, 1.0)
tau = a1 * tau1 + (1.0 - a1) * tau2
intensity = data_summed.sum(axis=-1)

tau1_ordered = tau1
tau2_ordered = tau2
a1_ordered = a1



thresholded_tau, thresholded_intensity, lifetime_mask = graph.threshold_lifetime_map(
    tau,
    intensity=intensity,
    threshold=0.05,
)

print("finite tau_eff pixels:", np.count_nonzero(np.isfinite(tau)))
print("finite tau1 pixels:", np.count_nonzero(np.isfinite(tau1)))
print("finite tau2 pixels:", np.count_nonzero(np.isfinite(tau2)))
print("thresholded pixels:", thresholded_tau.size)


In [ ]:
# RGB component-intensity image: red = tau1 component, blue = tau2 component
component_tau1_intensity = a1 * intensity
component_tau2_intensity = (1.0 - a1) * intensity

component_mask = (
    lifetime_mask
    & np.isfinite(component_tau1_intensity)
    & np.isfinite(component_tau2_intensity)
)

valid_intensity = intensity[component_mask]
if valid_intensity.size:
    component_vmax = np.nanpercentile(valid_intensity, 99.5)
else:
    component_vmax = np.nanpercentile(intensity[np.isfinite(intensity)], 99.5)

if not np.isfinite(component_vmax) or component_vmax <= 0:
    component_vmax = np.nanmax(intensity)

rgb_components = np.zeros((*intensity.shape, 3), dtype=float)
rgb_components[..., 0] = np.clip(component_tau1_intensity / component_vmax, 0.0, 1.0)
rgb_components[..., 2] = np.clip(component_tau2_intensity / component_vmax, 0.0, 1.0)
rgb_components[~component_mask] = 0.0

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(rgb_components, origin="lower")
ax.set_title("Biexponential component intensity: tau1 red, tau2 blue")
ax.set_axis_off()
plt.show()


In [ ]:
graph.plot_lifetime_summary(
    intensity=intensity,
    lifetime=tau,
    pxsize=pixel_size_x_um,
    pxdwelltime=pxdwelltime,
    lifetime_bounds=[2.5, 4.5],
    crop=30,
    threshold=0.05,
    bins=500,
    colormap="turbo",
    weighted_histogram=True,
)


In [ ]:
graph.plot_equalized_lifetime_summary(
    intensity=intensity,
    lifetime=tau,
    pxsize=pixel_size_x_um,
    pxdwelltime=pxdwelltime,
    lifetime_bounds=[1.0, 8.0],
    crop=30,
    threshold=0.05,
    bins=500,
    colormap="turbo",
    equalization_reference=thresholded_tau,
    equalization_strength=4.0,
    equalization_bins=4096,
    colorbar_ticks=12,
)

